In [1]:
import contextlib
import gc
import io
import os
import re

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from transformers import AutoModelForSequenceClassification, AutoTokenizer

try:
    from huggingface_hub.utils import disable_progress_bars
    from transformers.utils import logging as transformers_logging

    disable_progress_bars()
    transformers_logging.set_verbosity_error()
    transformers_logging.disable_progress_bar()
except Exception:
    pass


pd.set_option("display.max_colwidth", None)

text = "The product works well and I am very satisfied."

model_names = [
    "nlptown/bert-base-multilingual-uncased-sentiment",
    "distilbert-base-uncased-finetuned-sst-2-english",
    "cardiffnlp/twitter-roberta-base-sentiment",
    "microsoft/deberta-v3-base",
    "siebert/sentiment-roberta-large-english",
    "AnkitAI/reviews-roberta-base-sentiment-analysis",
]


In [2]:
SENTIMENT_VALUES = {
    "negative": -1.0,
    "neutral": 0.0,
    "positive": 1.0,
}

MODEL_NOTES = {
    "microsoft/deberta-v3-base": (
        "Base model, not sentiment fine-tuned; sequence-classification labels may be generic."
    ),
}


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def ordered_labels(config, num_labels):
    return [str(config.id2label.get(i, config.id2label.get(str(i), f"LABEL_{i}"))) for i in range(num_labels)]


def label_index(label):
    match = re.fullmatch(r"label[_ -]?(\d+)", str(label).strip().lower())
    return int(match.group(1)) if match else None


def sentiment_bucket(label, num_labels):
    cleaned = str(label).strip().lower()

    if any(token in cleaned for token in ["positive", "pos", "4 star", "4 stars", "5 star", "5 stars"]):
        return "positive"
    if any(token in cleaned for token in ["neutral", "3 star", "3 stars"]):
        return "neutral"
    if any(token in cleaned for token in ["negative", "neg", "1 star", "1 stars", "2 star", "2 stars"]):
        return "negative"

    idx = label_index(cleaned)
    if idx is None:
        return cleaned

    if num_labels == 2:
        return ["negative", "positive"][idx] if idx < 2 else cleaned
    if num_labels == 3:
        return ["negative", "neutral", "positive"][idx] if idx < 3 else cleaned
    if num_labels == 5:
        return ["negative", "negative", "neutral", "positive", "positive"][idx] if idx < 5 else cleaned

    return cleaned


def sentiment_value(label, num_labels, bucket):
    cleaned = str(label).strip().lower()
    star_match = re.search(r"\b([1-5])\s*stars?\b", cleaned)
    if star_match:
        return (int(star_match.group(1)) - 3) / 2

    idx = label_index(cleaned)
    if idx is not None:
        if num_labels == 2:
            return [-1.0, 1.0][idx] if idx < 2 else 0.0
        if num_labels == 3:
            return [-1.0, 0.0, 1.0][idx] if idx < 3 else 0.0
        if num_labels == 5:
            return [-1.0, -0.5, 0.0, 0.5, 1.0][idx] if idx < 5 else 0.0

    return SENTIMENT_VALUES.get(bucket, 0.0)


def round_probabilities(probabilities):
    return {label: round(float(probability), 4) for label, probability in probabilities.items()}


def analyze_model_sentiment(model_name, text, device):
    row = {
        "model_name": model_name,
        "input_text": text,
        "sentiment": pd.NA,
        "compound_score": pd.NA,
        "negative_probability": pd.NA,
        "neutral_probability": pd.NA,
        "positive_probability": pd.NA,
        "predicted_label": pd.NA,
        "predicted_label_probability": pd.NA,
        "label_probabilities": pd.NA,
        "status": "ok",
        "note": MODEL_NOTES.get(model_name, ""),
    }

    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
        model.eval()

        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        probabilities = F.softmax(logits, dim=-1).squeeze(0).detach().cpu()
        labels = ordered_labels(model.config, probabilities.numel())
        label_probabilities = dict(zip(labels, probabilities.tolist()))

        sentiment_probabilities = {"negative": 0.0, "neutral": 0.0, "positive": 0.0}
        compound_score = 0.0

        for label, probability in label_probabilities.items():
            bucket = sentiment_bucket(label, len(labels))
            if bucket in sentiment_probabilities:
                sentiment_probabilities[bucket] += probability
            compound_score += probability * sentiment_value(label, len(labels), bucket)

        top_index = int(torch.argmax(probabilities).item())
        sentiment = max(sentiment_probabilities, key=sentiment_probabilities.get)

        if all(label_index(label) is not None for label in labels):
            generic_note = "Generic labels mapped by label order."
            row["note"] = f"{row['note']} {generic_note}".strip()

        row.update(
            {
                "sentiment": sentiment,
                "compound_score": round(float(compound_score), 4),
                "negative_probability": round(sentiment_probabilities["negative"], 4),
                "neutral_probability": round(sentiment_probabilities["neutral"], 4),
                "positive_probability": round(sentiment_probabilities["positive"], 4),
                "predicted_label": labels[top_index],
                "predicted_label_probability": round(float(probabilities[top_index]), 4),
                "label_probabilities": round_probabilities(label_probabilities),
            }
        )

        del model, tokenizer
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as exc:
        model_note = MODEL_NOTES.get(model_name, "")
        row["status"] = "error"
        row["note"] = f"{model_note} Error: {exc}".strip()

    return row


In [3]:
device = get_device()
rows = []

for model_name in model_names:
    print(f"Analyzing sentiment with {model_name}...")
    rows.append(analyze_model_sentiment(model_name, text, device))

results_table = pd.DataFrame(rows)
results_table = results_table[
    [
        "model_name",
        "input_text",
        "sentiment",
        "compound_score",
        "negative_probability",
        "neutral_probability",
        "positive_probability",
        "predicted_label",
        "predicted_label_probability",
        "label_probabilities",
        "status",
        "note",
    ]
]

display(results_table)


Analyzing sentiment with nlptown/bert-base-multilingual-uncased-sentiment...


Analyzing sentiment with distilbert-base-uncased-finetuned-sst-2-english...


Analyzing sentiment with cardiffnlp/twitter-roberta-base-sentiment...


Analyzing sentiment with microsoft/deberta-v3-base...


Analyzing sentiment with siebert/sentiment-roberta-large-english...


Analyzing sentiment with AnkitAI/reviews-roberta-base-sentiment-analysis...


,model_name,input_text,sentiment,compound_score,negative_probability,neutral_probability,positive_probability,predicted_label,predicted_label_probability,label_probabilities,status,note
0,nlptown/bert-base-multilingual-uncased-sentiment,The product works well and I am very satisfied.,positive,0.7724,0.0041,0.0296,0.9663,5 stars,0.5843,"{'1 star': 0.0019, '2 stars': 0.0022, '3 stars': 0.0296, '4 stars': 0.382, '5 stars': 0.5843}",ok,
1,distilbert-base-uncased-finetuned-sst-2-english,The product works well and I am very satisfied.,positive,0.9997,0.0002,0.0,0.9998,POSITIVE,0.9998,"{'NEGATIVE': 0.0002, 'POSITIVE': 0.9998}",ok,
2,cardiffnlp/twitter-roberta-base-sentiment,The product works well and I am very satisfied.,positive,0.9852,0.0014,0.0121,0.9865,LABEL_2,0.9865,"{'LABEL_0': 0.0014, 'LABEL_1': 0.0121, 'LABEL_2': 0.9865}",ok,Generic labels mapped by label order.
3,microsoft/deberta-v3-base,The product works well and I am very satisfied.,NaN,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,error,"Base model, not sentiment fine-tuned; sequence-classification labels may be generic. Error: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`."
4,siebert/sentiment-roberta-large-english,The product works well and I am very satisfied.,positive,0.9979,0.0011,0.0,0.9989,POSITIVE,0.9989,"{'NEGATIVE': 0.0011, 'POSITIVE': 0.9989}",ok,
5,AnkitAI/reviews-roberta-base-sentiment-analysis,The product works well and I am very satisfied.,positive,0.9895,0.0052,0.0,0.9948,positive,0.9948,"{'negative': 0.0052, 'positive': 0.9948}",ok,
